In [1]:
import pandas as pd
from naturecubepy import (
    auth_headers,
    get_key,
    get_project,
    get_station_info,
    get_media_assets_df,
    set_segment_published_status,
)

In [2]:
i=5
countries = ['Brunei', 'Ecuador', 'Gabon', 'Germany', 'Peru', 'USA']
country_code = ['BRU', 'ECU', 'GAB', 'GER', 'PER', 'USA']
info = pd.read_excel('/Users/natimi/Downloads/Okala_birds_SFL_combined.xlsx')

In [3]:
# Retrieve API key and set up authentication headers
api_key = get_key(key_name='SL_'+country_code[i]+'_PROD')
hdr = auth_headers(api_key, okala_url="http://127.0.0.1:8000/api")
project_name = get_project(hdr)

Retrieving project data...
Received response with status code 200
Project data retrieved successfully
Setting your active project as - USA


In [4]:
# Build stations from already-fetched project metadata (avoids extra API call)
station_rows = []
for feat in project_name.locations.features:
    props = feat.properties
    station_rows.append({
        "project_system_record_id": getattr(props, "project_system_record_id", None),
        "measurement_type": str(getattr(props, "measurement_type", "")).strip().lower(),
        "device_id": getattr(props, "device_id", None),
    })

stations = pd.DataFrame(station_rows).dropna(subset=["project_system_record_id"])
stations = stations[stations["measurement_type"].isin(["bioacoustic", "camera"])].drop_duplicates(subset=["project_system_record_id"])
print(f"Loaded {len(stations)} stations from project metadata")

Loaded 12 stations from project metadata


In [8]:
# Get media data for selected station IDs
psr_ids = stations["project_system_record_id"].astype(int).tolist()

media_assets = get_media_assets_df(hdr, "audio", psr_ids[7:])
display(media_assets.head())

,label_id,label,common_name,class_,order,family,genus,species,iucn_redlist_status,tags,...,segment_record_id,label_record_id,prediction_accuracy,manager_verified,labeller_verified,blank,segment_record_published,segment_verification_status,device_id,data_type
0,49188,Geothlypis philadelphia,Mourning Warbler,Aves,Passeriformes,Parulidae,Geothlypis,Geothlypis philadelphia,Least Concern,[],...,795015,1898737,94.397797,False,False,False,True,ai_derived,0068W2WC2,audio
1,49188,Geothlypis philadelphia,Mourning Warbler,Aves,Passeriformes,Parulidae,Geothlypis,Geothlypis philadelphia,Least Concern,[],...,795017,1898739,85.349442,False,False,False,True,ai_derived,0068W2WC2,audio
2,49531,Limnothlypis swainsonii,Swainson's Warbler,Aves,Passeriformes,Parulidae,Limnothlypis,Limnothlypis swainsonii,Least Concern,[],...,794999,1898721,99.295905,False,False,False,True,ai_derived,0068W2WC2,audio
3,49390,Contopus virens,Eastern Wood-pewee,Aves,Passeriformes,Tyrannidae,Contopus,Contopus virens,Least Concern,[],...,795001,1898723,99.680415,False,False,False,True,ai_derived,0068W2WC2,audio
4,46325,Troglodytes aedon,House Wren,Aves,Passeriformes,Troglodytidae,Troglodytes,Troglodytes aedon,Least Concern,[],...,795003,1898725,75.194525,False,False,False,True,ai_derived,0068W2WC2,audio


In [9]:
subset = info[info.Country == countries[i]]

publish_map = subset.set_index('Species')['Publish Detections']
species_lookup = media_assets.species.map(publish_map)

unpublish_mask = (
    (species_lookup == 'none') |
    ((species_lookup == 'all above 95%') & (media_assets['prediction_accuracy'] < 95)) |
    ((species_lookup == 'only human verified') & (media_assets.segment_verification_status == 'ai_derived'))
)

In [10]:
for status, mask in [(False, unpublish_mask), (True, ~unpublish_mask)]:
    set_segment_published_status(hdr, published_status=status, segment_record_ids=media_assets.loc[mask, 'segment_record_id'])

Publish status updated successfully
Publish status updated successfully


In [29]:
segment_ids = (
    media_assets.loc[media_assets["device_id"] == sid, "segment_record_id"]
    .astype("int64")
    .map(int)
    .tolist()
)

r = set_segment_published_status(
    hdr,
    published_status=False,
    segment_record_ids=segment_ids
)
rification_status', 'prediction_accuracy', 'segment_record_published']]

,device_id,segment_verification_status,prediction_accuracy,segment_record_published
387,0068W1ARU1,ai_derived,96.679360,False
580,0068W1ARU1,ai_derived,78.133600,False
869,0068W1ARU1,ai_derived,75.591758,False
933,0068W1ARU1,ai_derived,99.946367,False
1129,0068W1ARU1,ai_derived,90.374313,False
...,...,...,...,...
26365,0068W2WC2,ai_derived,53.622344,False
26378,0068W2WC2,ai_derived,60.782720,False
26401,0068W3WC3,ai_derived,42.015843,False
26419,0068W3WC3,ai_derived,35.926343,False


In [15]:
seg_id = media_assets[(media_assets.species == 'Setophaga magnolia')]['segment_record_id']
r = set_segment_published_status(hdr, published_status=False, segment_record_ids=seg_id)

Publish status updated successfully
